In [5]:
# !pip install -q transformer-lens transformers accelerate

from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")

print("Model loaded!")
print(f"Layers: {model.cfg.n_layers}")
print(f"Hidden dimension: {model.cfg.d_model}")

/tmp/ipykernel_773/2080072307.py:5: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Model loaded!
Layers: 28
Hidden dimension: 1024


In [10]:
prompt = """Solve this problem step by step.

Alice is older than Bob.
Bob is older than Charlie.

Question: Who is the oldest?

Explain your reasoning and then give the final answer."""

tokens = model.to_tokens(prompt)

logits, cache = model.run_with_cache(tokens)

print("Number of tokens:", tokens.shape[1])
print("Cached activation keys:", len(cache))

Number of tokens: 38
Cached activation keys: 677


In [11]:
for layer in [0, 7, 14, 21, 27]:
    activation = cache[f"blocks.{layer}.hook_resid_post"]
    print(
        f"Layer {layer}:",
        activation.shape
    )

Layer 0: torch.Size([1, 38, 1024])
Layer 7: torch.Size([1, 38, 1024])
Layer 14: torch.Size([1, 38, 1024])
Layer 21: torch.Size([1, 38, 1024])
Layer 27: torch.Size([1, 38, 1024])


In [4]:
clean_prompt = """Alice is older than Bob.
Bob is older than Charlie.
Who is the oldest?"""

corrupt_prompt = """Alice is younger than Bob.
Bob is older than Charlie.
Who is the oldest?"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']

Corrupted:
['Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']


In [5]:
clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

clean_next = clean_logits[0, -1].argmax()
corrupt_next = corrupt_logits[0, -1].argmax()

print("Clean prediction:", repr(model.to_string(clean_next)))
print("Corrupted prediction:", repr(model.to_string(corrupt_next)))

Clean prediction: ' A'
Corrupted prediction: ' A'


In [6]:
clean_prompt = """Answer the question with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Alice"""

corrupt_prompt = """Answer the question with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Bob"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Alice']

Corrupted:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Bob']


In [10]:
clean_prompt = """Answer with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

corrupt_prompt = """Answer with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

# Probability of the two candidate answers
for name in [" Alice", " Bob"]:
    token = model.to_single_token(name)
    clean_prob = clean_logits[0, -1].softmax(-1)[token].item()
    corrupt_prob = corrupt_logits[0, -1].softmax(-1)[token].item()

    print(f"{name}: clean={clean_prob:.4f}, corrupted={corrupt_prob:.4f}")

 Alice: clean=0.1046, corrupted=0.0604
 Bob: clean=0.0052, corrupted=0.0048


## Outcome

The initial toy age-ordering task was not suitable for activation patching with Qwen3-0.6B: the model did not realiably distinguish the clean and corrupted conditions. We therefore do not interpret any activation patching results from this task.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
chat_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {
        "role": "user",
        "content": """Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

inputs = tokenizer(text, return_tensors="pt").to(chat_model.device)

outputs = chat_model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.6,
    top_p=0.95,
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

<think>
Okay, let's see. The problem says Alice is older than Bob, and Bob is older than Charlie. The question is asking who is the oldest. Hmm, so I need to figure out the hierarchy here.

First, let's break down the information given. Alice is older than Bob. That means Alice's age is higher than Bob's. Then Bob is older than Charlie. So Bob's age is higher than Charlie's. Now, if both of these statements are true, how does that affect the order of ages?

Let me visualize this. If Alice is older than Bob, then Alice comes first. Then Bob is older than Charlie, so Charlie comes after Bob. So putting it all together, the order from youngest to oldest would be Charlie, then Bob, then Alice. Wait, but does that make sense?

Wait, hold on. Let me think again. If Alice is older than Bob, then Alice is older than Charlie? No, because Charlie is only mentioned in the second statement. The first statement says Alice is older than Bob, but does that mean Alice is older than Charlie? The proble

## Qwen3 Chat-Template Sanity Check

The initial raw-text interface produced unreliable behavior on the toy
reasoning task. We therefore tested Qwen3-0.6B using its official chat
template with thinking mode enabled.

The chat-formatted model produced a coherent chain-of-thought trajectory
for the age-ordering task, suggesting that the earlier failure was at least
partly due to the inference interface rather than the model's inability to
solve the task.

Next: determine whether the same chat-formatted model can be connected to
our activation-inspection pipeline.

In [2]:
# Inspect the exact chat-formatted prompt we gave Qwen
print(text)

<|im_start|>user
Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer.<|im_end|>
<|im_start|>assistant



In [6]:
# Tokenize the chat-formatted prompt with TransformerLens
chat_tokens = model.to_tokens(text)

print("Number of tokens:", chat_tokens.shape[1])
print(model.to_str_tokens(chat_tokens))

Number of tokens: 35
['<|im_start|>', 'user', '\n', 'Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Ex', 'plain', ' your', ' reasoning', ' and', ' give', ' the', ' final', ' answer', '.', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n']


In [11]:
generated = model.generate(
    chat_tokens,
    max_new_tokens=500,
    temperature=0.6,
    top_p=0.95,
    verbose=False,
)

print(model.to_string(generated))

["<|im_start|>user\nAlice is older than Bob.\nBob is older than Charlie.\n\nWho is the oldest?\nExplain your reasoning and give the final answer.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see. The problem says Alice is older than Bob, and Bob is older than Charlie. I need to figure out who is the oldest among them. Hmm.\n\nSo, first, let's break down the information. Alice > Bob and Bob > Charlie. So starting from the top, Alice is the oldest, then Bob, then Charlie. So if we list them in order, it would be Alice, Bob, Charlie. That makes sense. But wait, maybe there's a trick here?\n\nWait, the question is about who is the oldest. The statements are comparing Alice to Bob and Bob to Charlie. So each of these comparisons is a direct comparison. So Alice is older than Bob, and Bob is older than Charlie. Therefore, the order from oldest to youngest is Alice, Bob, Charlie. So the answer should be Alice.\n\nBut let me double-check. Suppose Alice is 20, Bob is 15, and Charlie i

In [12]:
generated_for_cache = generated.clone().detach()

logits, cot_cache = model.run_with_cache(generated_for_cache)

print("Generated sequence tokens:", generated_for_cache.shape[1])
print("Cached activation tensors:", len(cot_cache))

Generated sequence tokens: 350
Cached activation tensors: 677


In [13]:
tokens = model.to_str_tokens(generated_for_cache)

for i, token in enumerate(tokens):
    print(f"{i:3d}: {repr(token)}")

  0: '<|im_start|>'
  1: 'user'
  2: '\n'
  3: 'Alice'
  4: ' is'
  5: ' older'
  6: ' than'
  7: ' Bob'
  8: '.\n'
  9: 'Bob'
 10: ' is'
 11: ' older'
 12: ' than'
 13: ' Charlie'
 14: '.\n\n'
 15: 'Who'
 16: ' is'
 17: ' the'
 18: ' oldest'
 19: '?\n'
 20: 'Ex'
 21: 'plain'
 22: ' your'
 23: ' reasoning'
 24: ' and'
 25: ' give'
 26: ' the'
 27: ' final'
 28: ' answer'
 29: '.'
 30: '<|im_end|>'
 31: '\n'
 32: '<|im_start|>'
 33: 'assistant'
 34: '\n'
 35: '<think>'
 36: '\n'
 37: 'Okay'
 38: ','
 39: ' let'
 40: "'s"
 41: ' see'
 42: '.'
 43: ' The'
 44: ' problem'
 45: ' says'
 46: ' Alice'
 47: ' is'
 48: ' older'
 49: ' than'
 50: ' Bob'
 51: ','
 52: ' and'
 53: ' Bob'
 54: ' is'
 55: ' older'
 56: ' than'
 57: ' Charlie'
 58: '.'
 59: ' I'
 60: ' need'
 61: ' to'
 62: ' figure'
 63: ' out'
 64: ' who'
 65: ' is'
 66: ' the'
 67: ' oldest'
 68: ' among'
 69: ' them'
 70: '.'
 71: ' Hmm'
 72: '.\n\n'
 73: 'So'
 74: ','
 75: ' first'
 76: ','
 77: ' let'
 78: "'s"
 79: ' break'
 8

In [14]:
# Test whether the corrupted prompt reliably changes the model's answer

corrupt_messages = [
    {
        "role": "user",
        "content": """Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

corrupt_text = tokenizer.apply_chat_template(
    corrupt_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

corrupt_inputs = tokenizer(
    corrupt_text,
    return_tensors="pt"
).to(chat_model.device)

n_samples = 10

for i in range(n_samples):
    corrupt_output = chat_model.generate(
        **corrupt_inputs,
        max_new_tokens=500,
        temperature=0.6,
        top_p=0.95,
    )

    response = tokenizer.decode(
        corrupt_output[0][corrupt_inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    print(response)


SAMPLE 1
<think>
Okay, let's see. The problem says Alice is younger than Bob, and Bob is older than Charlie. I need to figure out who is the oldest. Hmm.

So, first, let's break down the given information. Alice is younger than Bob. That means Alice's age is less than Bob's. So Bob is older than Alice. Then, Bob is older than Charlie. So Charlie's age is less than Bob's age. Therefore, Charlie is younger than Bob. 

Wait, so if Alice is younger than Bob, and Bob is older than Charlie, then Charlie is younger than Bob. But the question is asking who is the oldest. So between Bob and Charlie, Bob is older. But wait, is there any other possibility? Maybe Alice is older than Charlie? Wait, no, the problem says Alice is younger than Bob. So Alice < Bob. And Bob > Charlie. So Charlie < Bob. So Charlie is the youngest. Bob is the oldest. Therefore, the answer should be Bob. 

Let me check again. If Alice < Bob, then Bob is older than Alice. Bob is older than Charlie, so Charlie is older than

## Corrupted Prompt Reliability Check

We tested the corrupted version of the toy task across 10 independent
samples using temperature 0.6 and top-p 0.95.

The corrupted prompt changed the first relationship from:

Alice > Bob

to:

Alice < Bob

while keeping:

Bob > Charlie

All 10 samples concluded that Bob is the oldest.

Thus, for this toy task, the intervention produces a reliable behavioral
contrast between the clean and corrupted conditions.